<a href="https://colab.research.google.com/github/matthewpecsok/IS4490_student_course_files/blob/main/IS4490_module_10_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Module 10 Assignment: Three Questions, One Grounded Pipeline

Use the annual report PDF to write three questions based on your curiosity:

1. A two-part question whose answers require evidence from two different parts of the report.
2. A focused question with a specific numeric answer stated in the report.
3. A broad question about the report as a whole.

For **each** question, run the same three steps:

1. **Similarity search:** retrieve the top 20 candidate chunks.
2. **Reranking:** use the cross-encoder to select the top 3 chunks.
3. **Answer generation:** send the question and selected chunks to the language model.

Check the generated answer against the original PDF. Record the question, answer, relevant retrieved chunks, page evidence, and your assessment before moving to the next question. The notebook reuses the query and result variables, so later runs replace the displayed outputs. The later prompt-cache cells are demonstrations, not additional assignment runs.

In [ ]:
!pip install -q langchain langchain-community pypdf chromadb sentence-transformers

import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


/tmp/ipykernel_1225/1468499213.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
import json

In [ ]:
# Colab setup step 1: install zstd, which the Ollama installer expects on Ubuntu.
!apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 33 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 2s (326 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


# Download Ollama

In [ ]:
# Colab setup step 2: install Ollama.
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


# Start Ollama

### RESTART ME IF NEEDED

In [ ]:
# Colab setup step 3: start the Ollama server and pull the model.
import subprocess
import time

ollama_server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)

pull = subprocess.run(["ollama", "pull", "gemma4:12b"], capture_output=True, text=True)
print(pull.stdout[-1000:])
print(pull.stderr[-1000:])

pull = subprocess.run(["ollama", "pull", "qwen3:14b"], capture_output=True, text=True)
print(pull.stdout[-1000:])
print(pull.stderr[-1000:])

print("Ollama setup attempted. If the pull failed, check the runtime log and rerun this cell.")


ng 0cb0cd0a5c49:  99% ▕█████████████████ ▏ 173 MB/175 MB  154 MB/s      0s
pulling bb722270d543: 100% ▕█████████████████ ▏ 7.4 GB/7.4 GB  187 MB/s      0s
pulling 7008a656050b: 100% ▕██████████████████▏ 465 MB                         
pulling b507b9c2f6ca: 100% ▕██████████████████▏   13 B                         
pulling e5ca99f91896: 100% ▕██████████████████▏  242 B                         
verifying sha256 digest ⠴ pulling manifest 
pulling 0cb0cd0a5c49:  99% ▕█████████████████ ▏ 173 MB/175 MB  154 MB/s      0s
pulling bb722270d543: 100% ▕█████████████████ ▏ 7.4 GB/7.4 GB  187 MB/s      0s
pulling 7008a656050b: 100% ▕██████████████████▏ 465 MB                         
pulling b507b9c2f6ca: 100% ▕██████████████████▏   13 B                         
pulling e5ca99f91896: 100% ▕██████████████████▏  242 B                         
verifying sha256 digest 
writing manifest 
success 


1361f314: 100% ▕█████████████████ ▏ 9.3 GB/9.3 GB  163 MB/s      0s
pulling 78b3b822087d: 100% ▕██████████

In [ ]:
!ollama --version

ollama version is 0.40.13


In [ ]:
# Colab setup step 4: install Python packages.
!pip install langchain langchain-ollama --quiet

In [ ]:
model = "qwen3:14b"

You might be wondering why we aren't using the Gemma model this time. It can also use prompt caching, but the sliding context window makes it less able to reuse the prompt cache.

Prompt caching allows the model to skip parts of the prompt processing. It works like this

**PROMPT CACHING:** if the model receives a request that STARTS with something it has seen before it can skip reprocessing that section which enables it to process the request faster.

2)

## Setup LangChain with the qwen model

In [ ]:
from datetime import datetime

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

llm = ChatOllama(model=model, temperature=0)

print(f"Ready. Using {model} through Ollama and LangChain.")

Ready. Using qwen3:14b through Ollama and LangChain.


## Test the LLM with a simple prompt

First time this takes about 2-3 minutes.

In [ ]:
llm.invoke("what is the capitol of france?")

AIMessage(content='The capital of France is **Paris**. It is a major global city known for its cultural landmarks, such as the Eiffel Tower, the Louvre Museum, and Notre-Dame Cathedral. Paris has been the political, economic, and cultural heart of France for centuries.', additional_kwargs={}, response_metadata={'model': 'qwen3:14b', 'created_at': '2026-11-01T02:09:36.937442065Z', 'done': True, 'done_reason': 'stop', 'total_duration': 172408738122, 'load_duration': 102325331086, 'prompt_eval_count': 18, 'prompt_eval_duration': 45072985000, 'eval_count': 200, 'eval_duration': 25004040000, 'logprobs': None, 'model_name': 'qwen3:14b', 'model_provider': 'ollama'}, id='lc_run--01a194db-f11a-735a-88bb-d9053e0e466c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 200, 'total_tokens': 218})

# Retrieval Augmented Generation

## Download the PDF

In [ ]:
!wget "https://github.com/matthewpecsok/IS4490_student_course_files/raw/main/250219-annual-report-and-accounts-2024.pdf" -O 250219-annual-report-and-accounts-2024.pdf

--2026-11-01 02:09:36--  https://github.com/matthewpecsok/IS4490_student_course_files/raw/main/250219-annual-report-and-accounts-2024.pdf
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/matthewpecsok/IS4490_student_course_files/main/250219-annual-report-and-accounts-2024.pdf [following]
--2026-11-01 02:09:37--  https://raw.githubusercontent.com/matthewpecsok/IS4490_student_course_files/main/250219-annual-report-and-accounts-2024.pdf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8670552 (8.3M) [application/octet-stream]
Saving to: ‘250219-annual-report-and-accounts-2024.pdf’

250219-annual-

# Chunk, Embed and Store

This code block performs the following tasks. It only needs to run once for the document.

1. Load the PDF
2. Chunk the document
3. Add custom metadata to the chunks
4. Initialize embedding model
5. Store chunks into a local Chroma vector database

In [ ]:
# 1. Load the PDF
pdf_path = "/content/250219-annual-report-and-accounts-2024.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# 2. Chunk the documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
chunks = text_splitter.split_documents(documents)

# 3. Add custom metadata to the chunks as requested
for i, chunk in enumerate(chunks):
    chunk.metadata["source_file"] = "250219-annual-report-and-accounts-2024.pdf"
    chunk.metadata["chunk_index"] = i

# 4. Initialize embedding model
# Using a lightweight, free huggingface embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 5. Store chunks into a local Chroma vector database
persist_directory = "./chroma_db"
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory
)

print(f"Successfully loaded and chunked {len(documents)} pages into {len(chunks)} text chunks.")
print(f"Stored vector database locally in '{persist_directory}'.")

/tmp/ipykernel_1225/3131070098.py:20: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Successfully loaded and chunked 460 pages into 2711 text chunks.
Stored vector database locally in './chroma_db'.


# RUN AGAIN STEP 1 - Similarity Search

Take a given question and perform a similarity search on the vector database.

Return the top 20 results.

In [ ]:
query = "Replace this text with your first question about the annual report."


similarity_k=20 # return top k results

candidate_results = vector_store.similarity_search(query, k=similarity_k)

# Display the results
for idx, doc in enumerate(candidate_results, 1):
    print(f"--- MATCH {idx} ---")
    print(f"Source File: {doc.metadata.get('source_file')}")
    print(f"Chunk Index: {doc.metadata.get('chunk_index')}")
    print(f"Page Number: {doc.metadata.get('page')}")
    print(f"Content snippet:\n{doc.page_content[:400]}...")
    print("\n")

# RUN AGAIN STEP 2 - Reranking cross-encoder

Rerank the top 20 chunks using a cross-encoding model. Usually a small model fast, and cheap to run. It predicts the similarity of two chunks (the question and the possible matched chunk). The new score is then used to find the best chunks.

In [ ]:
from sentence_transformers import CrossEncoder

# 1. Load a pre-trained cross-encoder reranker model
# ms-marco-MiniLM-L-6-v2 is a popular, fast, and high-performing model for reranking
reranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# 2. Prepare pairs of (query, document_text) for the cross-encoder
pairs = [[query, doc.page_content] for doc in candidate_results]

# 3. Compute relevance scores
scores = reranker_model.predict(pairs)

# 4. Pair each candidate with its score and sort by score descending
reranked_results = sorted(zip(candidate_results, scores), key=lambda x: x[1], reverse=True)

n = 3
print(f"=== TOP {n} RERANKED MATCHES ===\n")
for idx, (doc, score) in enumerate(reranked_results[:n], 1):
    print(f"--- RERANKED MATCH {idx} (Score: {score:.4f}) ---")
    print(f"Source File: {doc.metadata.get('source_file')}")
    print(f"Chunk Index: {doc.metadata.get('chunk_index')}")
    print(f"Page Number: {doc.metadata.get('page')}")
    print(f"Content snippet:\n{doc.page_content[:400]}...")
    print("\n")

Remind ourselves of the query (question)

In [ ]:
# you can update the query here.
#query = ""

In [ ]:
query

'How many years has Kalpana Morparia been in banking?'

Pass the query and the reranked results

In [ ]:
from IPython.display import display, Markdown

# RUN AGAIN STEP 3 - Send reranks to LLM

Notice below we aren't using LangChain, unfortunantely the langollama package doesn't allow us to see cache usage, so we go straight to the ollama service api.

In [ ]:
import requests
import json

# Define the local Ollama generation API endpoint
url = "http://localhost:11434/api/generate"

# Construct the payload requesting the gemma4:12b model
payload = {
    "model": model,
    "prompt": str(reranked_results[:n])+query,
    "stream": False
}

# Make the POST request
try:
    response = requests.post(url, json=payload)
    response.raise_for_status()
    result = response.json()
    print("Ollama Response:")
    print(display(Markdown(result.get("response"))))
except Exception as e:
    print(f"An error occurred while calling the API: {e}")

import json



In [ ]:
# Convert response.content raw bytes directly to a dictionary
response_dict_parsed = json.loads(response.content)
print(f"The model used {response_dict_parsed.get("prompt_eval_cached_count")} cached tokens")

If we reverse the prompt text so that the model has not seen the prompt prefix, is the model able to use cache?

Spoiler alert. It should not.

In [ ]:
import requests
import json

# Define the local Ollama generation API endpoint
url = "http://localhost:11434/api/generate"

# Construct the payload requesting the gemma4:12b model
payload = {
    "model": model,
    "prompt": query+str(reranked_results[:n]),
    "stream": False
}

# Make the POST request
try:
    response = requests.post(url, json=payload)
    response.raise_for_status()
    result = response.json()
    print("Ollama Response:")
    print(display(Markdown(result.get("response"))))
except Exception as e:
    print(f"An error occurred while calling the API: {e}")

import json



In [ ]:
# Convert response.content raw bytes directly to a dictionary
response_dict_parsed = json.loads(response.content)
print(f"The model used {response_dict_parsed.get("prompt_eval_cached_count")} cached tokens")

Rerun the question again, with a slight wording change

In [ ]:
query = "For what number of years has Kalpana Morparia been in banking?"

In [ ]:
import requests
import json

# Define the local Ollama generation API endpoint
url = "http://localhost:11434/api/generate"

# Construct the payload requesting the gemma4:12b model
payload = {
    "model": model,
    "prompt": str(reranked_results[:n])+query,
    "stream": False
}

# Make the POST request
try:
    response = requests.post(url, json=payload)
    response.raise_for_status()
    result = response.json()
    print("Ollama Response:")
    print(display(Markdown(result.get("response"))))
except Exception as e:
    print(f"An error occurred while calling the API: {e}")

import json



In [ ]:
# Convert response.content raw bytes directly to a dictionary
response_dict_parsed = json.loads(response.content)
print(f"The model used {response_dict_parsed.get("prompt_eval_cached_count")} cached tokens")